# 05 · Troubleshooting

Each section reproduces a problem that users meet, shows the message or the
symptom, explains the cause and gives the fix. Some problems raise an error; others
are worse, because the detection runs and returns wrong results without a warning.

| # | Symptom | Cause |
|---|---|---|
| 1 | `UserWarning: xrMHW needs the whole time axis in one chunk` | time split across Dask chunks |
| 2 | `ValueError: Input dataset must have a 'time' coordinate` | the time dimension has another name |
| 3 | `ValueError: No data in the climatology baseline`, raised only at `.compute()`; or no error and a shorter baseline | baseline outside the record, or partly outside |
| 4 | climatology of about 290 "°C" | temperature in kelvin |
| 5 | no error, wrong durations | missing dates |
| 6 | no error, meaningless events | data that are not daily |
| 7 | no error, shifted dates and wrong events | `noleap` or `360_day` calendar |
| 8 | the machine runs out of memory | chunks too large, or all outputs kept in memory |
| 9 | netCDF error when writing with a Dask cluster | parallel netCDF writes from processes |
| 10 | outputs are all NaN in some cells | cells with no data, or too many gaps |

In [1]:
import numpy as np
import pandas as pd
import pooch
import xarray as xr

import xrMHW

EXAMPLE_DATA = pooch.create(
    path=pooch.os_cache("xrmhw"),
    base_url="doi:10.5281/zenodo.22909109/",
    registry={
        "xrmhw_glorys_4d_subset.nc": "md5:5555737e45f1c6cde9f657e50f456693",
        "xrmhw_ostia_sst_3d.nc": "md5:74a2da227394e4463f203f6b388518ee",
    },
)
glorys = xr.open_dataset(EXAMPLE_DATA.fetch("xrmhw_glorys_4d_subset.nc"))

# A surface series and a small 3-D block from the example, used throughout
series = glorys[["thetao"]].sel(latitude=44.0, longitude=-48.75, method="nearest"
                                ).isel(depth=0).load()
block = glorys[["thetao"]].isel(depth=0, latitude=slice(0, 4), longitude=slice(0, 4)).load()
CLIM = (1993, 2022)
reference = xrMHW.xrMHW_func(series, "thetao", clim_period=CLIM).load()


def n_event_days(out, prefix="mhw"):
    return int(out[f"{prefix}_severity"].notnull().sum())


print(f"reference run: {n_event_days(reference)} marine heatwave days")

reference run: 918 marine heatwave days


## 1. Time is split across chunks

**Symptom.**

In [2]:
import warnings

# Record the warning, to print it here
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    split = xrMHW.xrMHW_func(block.chunk({"time": 365}), "thetao", clim_period=CLIM)
for w in caught:
    print(f"{w.category.__name__}: {w.message}")
print("chunks of the result:", split["mhw_severity"].data.chunksize)

chunks of the result: (4, 4, 12017)


**Cause.** Each cell needs its whole record to compute its climatology, and Dask
was given the record in pieces. This happens after `open_mfdataset` (one chunk per
file) or `chunks={"time": ...}`. xrMHW joins the pieces into one chunk along time
and carries on, so the results are correct, but it leaves the other dimensions as
they were: the new chunks can be much larger than you planned, and so can the
memory they need (section 8). Earlier versions of xrMHW stopped here with
`ValueError: dimension time ... consists of multiple chunks`.

**Fix.** Chunk the input yourself: time whole, and the other dimensions split to
fit in memory (notebook 04 has a helper that sizes them).

In [3]:
ok = xrMHW.xrMHW_func(block.chunk({"time": -1, "latitude": 2}), "thetao", clim_period=CLIM)
print("works; chunks:", ok["mhw_severity"].data.chunksize)

works; chunks: (2, 4, 12017)


## 2. The time dimension has another name

**Symptom.**

In [4]:
renamed = series.rename(time="time_counter")          # as in NEMO output
try:
    xrMHW.xrMHW_func(renamed, "thetao", clim_period=CLIM)
except ValueError as err:
    print("ValueError:", err)

ValueError: Input dataset must have a 'time' coordinate/index.


**Cause.** xrMHW looks for a coordinate called `time`. Model output often uses
`time_counter`, `t` or `ocean_time`.

**Fix.** Rename it. Other dimensions can keep any name.

In [5]:
fixed = renamed.rename(time_counter="time")
print(n_event_days(xrMHW.xrMHW_func(fixed, "thetao", clim_period=CLIM)), "event days")

918 event days


## 3. The baseline is outside the record

**Symptom.** The call returns at once, because xrMHW only builds a lazy computation
on Dask arrays. The error appears when the values are computed, which can be much
later, for example when writing to disk:

In [6]:
lazy = xrMHW.xrMHW_func(block.chunk({"time": -1}), "thetao", clim_period=(1961, 1990))
print("the call returned without error")
try:
    lazy.compute()
except ValueError as err:
    print("ValueError at compute():", err)

the call returned without error


ValueError at compute(): No data in the climatology baseline 1961-1990; the record spans 1993-2025.


**Cause.** The years in `clim_period` must be within the record; here the data
start in 1993.

**Fix.** Choose a baseline inside the record, of 30 years if possible. Check the
record first with `ds.time.dt.year.min().item()` and `...max().item()`. Different
baselines give different events; Smith et al. (2025) discuss how to choose one.

A baseline that only **partly** overlaps the record raises no error: xrMHW uses the
years it has. Here 1982–2011 on a record that starts in 1993 silently becomes a
19-year baseline, 1993–2011:

In [7]:
partial = xrMHW.xrMHW_func(series, "thetao", clim_period=(1982, 2011)).load()
print(f"clim_period=(1982, 2011): {n_event_days(partial)} event days, against "
      f"{n_event_days(reference)} with 1993-2022")

clim_period=(1982, 2011): 1001 event days, against 918 with 1993-2022


## 4. Temperature in kelvin

**Symptom.** No error. Intensity and severity are right, but the climatology and
threshold are about 273 larger and labelled °C:

In [8]:
kelvin = series.copy()
kelvin["thetao"] = kelvin["thetao"] + 273.15
out_k = xrMHW.xrMHW_func(kelvin, "thetao", clim_period=CLIM).load()
print(f"climatology: {float(out_k.climatology.mean()):.1f} "
      f"{out_k.climatology.attrs['units']}")
print("same events as in °C:", n_event_days(out_k) == n_event_days(reference))
print(f"max severity difference: "
      f"{float(abs(out_k.mhw_severity - reference.mhw_severity).max()):.1e}")

climatology: 280.4 degC
same events as in °C: True
max severity difference: 0.0e+00


**Cause.** Detection depends only on differences, which are the same in K and °C,
but the output attributes assume °C.

**Fix.** Convert to °C before the detection (`ds["sst"] - 273.15`). Decide from the
values, not only from the `units` attribute, which can be wrong (the OSTIA example
says kelvin but holds °C, see notebooks 00 and 02).

## 5. Missing dates

**Symptom.** No error, and the output looks plausible. Here, 10% of the days were
removed from the series, as happens when files are missing from a download:

In [9]:
rng = np.random.default_rng(0)
keep = np.sort(rng.choice(series.sizes["time"], int(0.9 * series.sizes["time"]),
                          replace=False))
gappy = series.isel(time=keep)
out_gappy = xrMHW.xrMHW_func(gappy, "thetao", clim_period=CLIM).load()
print(f"{n_event_days(out_gappy)} event days, against {n_event_days(reference)} "
      "with the complete record")

800 event days, against 918 with the complete record


**Cause.** xrMHW counts time steps, not days: the 5-day minimum, the 2-day gap and
the durations all assume one step per day. With dates missing, five steps can span
a week or more, and two separate events can look consecutive.

**Fix.** Check that the steps are all one day, and reindex onto a complete daily
axis. Missing days become NaN; runs of up to `max_gap_interp` days (2 by default)
are then filled by interpolation, and longer gaps stay missing.

In [10]:
steps = np.unique(np.diff(gappy.time.values).astype("timedelta64[D]").astype(int))
print("time steps found (days):", steps[:6], "..." if steps.size > 6 else "")

full_axis = pd.date_range(gappy.time.values[0], gappy.time.values[-1], freq="D")
regular = gappy.reindex(time=full_axis)
print(f"after reindexing: {int(regular.thetao.isnull().sum())} missing days to fill "
      f"or leave missing")
out_regular = xrMHW.xrMHW_func(regular, "thetao", clim_period=CLIM).load()
print(f"{n_event_days(out_regular)} event days (complete record: "
      f"{n_event_days(reference)})")

time steps found (days): [1 2 3 4] 
after reindexing: 1202 missing days to fill or leave missing


932 event days (complete record: 918)


The fixed run is close to the reference but not identical: days that are really
missing cannot be recovered, and gaps longer than two days stay missing.

## 6. Data that are not daily

**Symptom.** No error. Monthly or 5-day means run, but the 11-day window becomes 11
months or 55 days, and the 5-day minimum becomes 5 months.

**Cause.** The Hobday et al. (2016) definition, and xrMHW, are for daily data.

**Fix.** Check the frequency first. Use daily data, or, for sub-daily data, daily
means (`ds.resample(time="1D").mean()`).

In [11]:
monthly = series.resample(time="MS").mean()
print("frequency of the example:", xr.infer_freq(series.time))
print("frequency after monthly means:", xr.infer_freq(monthly.time), "-> not suitable")

frequency of the example: D
frequency after monthly means: MS -> not suitable


## 7. Model calendars: `noleap` and `360_day`

**Symptom.** No error. Climate model output often uses a 365-day (`noleap`) or
360-day calendar, with `cftime` dates. xrMHW 3.0.0 turns dates into day numbers
(ordinals) and reads them back as dates of the standard calendar, without checking
the calendar first. The day numbers of other calendars do not match, so every date
is read as another date.

In [12]:
from datetime import date

noleap = series.convert_calendar("noleap")                   # drops 29 February
day360 = series.convert_calendar("360_day", align_on="year")
for name, ds_cal in [("noleap", noleap), ("360_day", day360)]:
    t = ds_cal.time.values
    seen = [date.fromordinal(x.toordinal()) for x in (t[0], t[-1])]
    out_cal = xrMHW.xrMHW_func(ds_cal, "thetao", clim_period=CLIM).load()
    print(f"{name:8s} record {t[0].strftime('%Y-%m-%d')} to {t[-1].strftime('%Y-%m-%d')} "
          f"is read as {seen[0]} to {seen[1]}; {n_event_days(out_cal)} event days "
          f"(correct: {n_event_days(reference)})")

noleap   record 1993-01-01 to 2025-11-25 is read as 1992-09-04 to 2025-07-21; 903 event days (correct: 918)
360_day  record 1993-01-01 to 2025-11-24 is read as 1965-05-24 to 1997-10-26; 3043 event days (correct: 918)


On `noleap` the whole record moves back by about four months, so the seasonal
cycle and the baseline years are misplaced. On `360_day` the record appears to end
in 1997, the 1993–2022 baseline keeps only five years, and the event count triples.

**Cause.** Day numbers of a 365- or 360-day calendar do not match those of the
standard calendar.

**Fix.** Convert to the standard calendar before running xrMHW. The days that do
not exist in the model calendar (29 February, or the 31st of some months) become NaN
and are filled by interpolation, because they are isolated.

In [13]:
converted = noleap.convert_calendar("standard", missing=np.nan)
out_conv = xrMHW.xrMHW_func(converted, "thetao", clim_period=CLIM).load()
print(f"noleap -> standard: {int(converted.thetao.isnull().sum())} inserted days; "
      f"max climatology difference from the correct run "
      f"{float(abs(out_conv.climatology - reference.climatology).max()):.3f} °C")

conv360 = day360.convert_calendar("standard", align_on="date", missing=np.nan)
out_conv360 = xrMHW.xrMHW_func(conv360, "thetao", clim_period=CLIM).load()
print(f"360_day -> standard: runs, {n_event_days(out_conv360)} event days")

noleap -> standard: 8 inserted days; max climatology difference from the correct run 0.001 °C


360_day -> standard: runs, 905 event days


## 8. Running out of memory

**Symptom.** The machine slows down and swaps, the kernel dies, or Dask reports
workers exceeding their memory limit.

**Cause.** Peak memory is about 7.25 times the decoded size of the data processed at
once. Without chunks, that is the whole dataset. With a cluster, it is the number of
workers times the chunk size. Asking for all eight outputs and calling `.compute()`
also keeps eight arrays of the input's size in memory.

**Fix.**

- Estimate before running: `7.25 * ds[var].nbytes`.
- Chunk over space so that chunk ≤ memory per worker / 7.25 (notebook 04 has a
  helper), and use fewer workers if needed.
- Cast to `float32`, and ask only for the outputs you need (`output_vars`).
- Write the result to Zarr instead of calling `.compute()` on it.

In [14]:
nbytes = glorys["thetao"].astype("float32").nbytes
print(f"GLORYS example as float32: {nbytes / 2**30:.2f} GiB -> about "
      f"{7.25 * nbytes / 2**30:.1f} GiB peak if processed in one piece")

GLORYS example as float32: 0.21 GiB -> about 1.5 GiB peak if processed in one piece


## 9. netCDF errors when writing with a Dask cluster

**Symptom.** `to_netcdf` fails part-way with an error such as `NetCDF: HDF error` or
`OSError: [Errno -51] NetCDF: Unknown file format`, when the result is computed by a
`LocalCluster` of worker processes.

**Cause.** netCDF-4/HDF5 files are not designed for simultaneous writes from
several processes; xarray serialises the writes with locks, and large writes can
still fail.

**Fix.** Write to Zarr (`result.to_zarr(path)`), which stores each chunk separately
and needs no locks, and convert afterwards if you need netCDF. Or, for results that
fit in memory, `.compute()` first and then `to_netcdf`. Notebook 04 shows the Zarr
route.

## 10. Some cells are NaN everywhere

**Symptom.** Climatology, threshold and every event field are NaN in some cells.

**Cause.** Those cells have no data for the whole record: land, cells below the sea
floor, or areas masked in the source. xrMHW returns NaN for them without spending
time on them. Cells with long gaps keep NaN over the gaps, and a climatology day with
no valid data in the baseline is NaN too.

**Fix.** Nothing to do for land and sea floor. For gappy cells, check how much data
the baseline holds:

In [15]:
valid_share = glorys["thetao"].isel(depth=-1).sel(time=slice("1993", "2022")).notnull().mean("time")
print(f"deepest level: {int((valid_share == 0).sum())} cells without data "
      f"(sea floor), {int(((valid_share > 0) & (valid_share < 0.9)).sum())} cells with "
      "less than 90% of the baseline")

deepest level: 75 cells without data (sea floor), 0 cells with less than 90% of the baseline


## Still stuck?

Open an issue at https://github.com/Gabo2000s/xarray_MHW-xrMHW-/issues with the
output of `xrMHW.__version__`, `xr.show_versions()`, the call you ran and the full
error message. A small dataset that reproduces the problem helps most.

### References

- Hobday, A. J., et al. (2016). A hierarchical approach to defining marine heatwaves. *Progress in Oceanography*, 141, 227–238. https://doi.org/10.1016/j.pocean.2015.12.014
- Smith, K. E., et al. (2025). Baseline matters: Challenges and implications of different marine heatwave baselines. *Progress in Oceanography*, 231, 103404. https://doi.org/10.1016/j.pocean.2024.103404